# 03 — Model Training, Evaluation, and Export

This notebook trains the final deployable complaint classification model. TF-IDF is created inside this notebook as a sparse matrix and saved as a vectorizer `.pkl`, not as CSV.

In [9]:
# 03_MODEL_TRAINING_EVALUATION_AND_EXPORT.ipynb
# Customer Complaint Classification System

import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

# Project paths
BASE_DIR = r"C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT"

OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
MODEL_DIR = os.path.join(BASE_DIR, "models")
REPORT_DIR = os.path.join(BASE_DIR, "reports")

for folder in [OUTPUT_DIR, MODEL_DIR, REPORT_DIR]:
    os.makedirs(folder, exist_ok=True)

PROCESSED_FILE = os.path.join(OUTPUT_DIR, "02_processed_complaints.csv")

print("Processed file:", PROCESSED_FILE)


Processed file: C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\outputs\02_processed_complaints.csv


In [10]:
# Load processed data
df = pd.read_csv(PROCESSED_FILE)

print("Loaded shape:", df.shape)
df.head()


Loaded shape: (20928, 6)


,complaint_text,clean_text,category,raw_char_length,raw_word_length,clean_word_length
0,Good morning my name is XXXX XXXX and I appreciate it if you could help me put a stop to Chase Bank cardmember services. \nIn 2018 I wrote to Chase asking for debt verification and what they sent ...,good morning name xxxx xxxx appreciate could help put stop chase bank cardmember service write chase ask debt verification sent statement acceptable ask bank validate debt instead receive mail eve...,Debt collection,486,92,47
1,I upgraded my XXXX XXXX card in XX/XX/2018 and was told by the agent who did the upgrade my anniversary date would not change. It turned the agent was giving me the wrong information in order to u...,upgraded xxxx xxxx card told agent upgrade anniversary date would change turn agent give wrong information order upgrade account xxxx change anniversary date xxxx xxxx without consent xxxx record ...,Credit card or prepaid card,355,63,31
2,"Chase Card was reported on XX/XX/2019. However, fraudulent application have been submitted my identity without my consent to fraudulently obtain services. Do not extend credit without verifying th...",chase card report however fraudulent application submit identity without consent fraudulently obtain service extend credit without verify identity applicant,"Credit reporting, credit repair services, or other personal consumer reports",224,32,19
3,"On XX/XX/2018, while trying to book a XXXX XXXX ticket, I came across an offer for {$300.00} to be applied towards the ticket if I applied for a rewards card. I put in my information for the off...",try book xxxx xxxx ticket come across offer apply towards ticket apply reward card put information offer within less minute notify via screen decision could make immediately contact xxxx refer cha...,"Credit reporting, credit repair services, or other personal consumer reports",1502,269,119
4,my grand son give me check for {$1600.00} i deposit it into my chase account after fund clear my chase bank closed my account never paid me my money they said they need to speek with my grand son ...,grand son give check deposit chase account fund clear chase bank close account never paid money say need speek grand son check clear money take chase bank refuse pay money grand son call chase tim...,Checking or savings account,477,95,51


In [11]:
# Basic checks

required_columns = ["clean_text", "category"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df.dropna(subset=["clean_text", "category"])
df = df[df["clean_text"].str.strip() != ""]
df = df.reset_index(drop=True)

print("Shape after final safety check:", df.shape)
print("Number of categories:", df["category"].nunique())
df["category"].value_counts()


Shape after final safety check: (20928, 6)
Number of categories: 10


category
Credit card or prepaid card                                                     7099
Checking or savings account                                                     5936
Mortgage                                                                        3246
Credit reporting, credit repair services, or other personal consumer reports    2032
Debt collection                                                                  930
Money transfer, virtual currency, or money service                               853
Vehicle loan or lease                                                            638
Student loan                                                                     140
Payday loan, title loan, or personal loan                                         45
Other financial service                                                            9
Name: count, dtype: int64

In [12]:
# remove very rare categories
# This avoids training errors and improves reliability.
# we can increase min_samples_per_class if needed.

# Load fresh data
df_original = pd.read_csv(PROCESSED_FILE)

# 2. Remove Rare Categories
MIN_SAMPLES_PER_CLASS = 50

category_counts = df["category"].value_counts()

valid_categories = category_counts[category_counts >= MIN_SAMPLES_PER_CLASS].index

df_filtered = df[df["category"].isin(valid_categories)].copy()

df_filtered.reset_index(drop=True, inplace=True)

print("Minimum samples per class:", MIN_SAMPLES_PER_CLASS)
print("Shape after filtering:", df_filtered.shape)
print("Number of categories:", df_filtered["category"].nunique())

print("\nFinal category distribution:")
print(df_filtered["category"].value_counts())

Minimum samples per class: 50
Shape after filtering: (20874, 6)
Number of categories: 8

Final category distribution:
category
Credit card or prepaid card                                                     7099
Checking or savings account                                                     5936
Mortgage                                                                        3246
Credit reporting, credit repair services, or other personal consumer reports    2032
Debt collection                                                                  930
Money transfer, virtual currency, or money service                               853
Vehicle loan or lease                                                            638
Student loan                                                                     140
Name: count, dtype: int64


In [13]:
# Define X and y

X = df["clean_text"]
y = df["category"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Encoded classes:")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)


Encoded classes:
0 -> Checking or savings account
1 -> Credit card or prepaid card
2 -> Credit reporting, credit repair services, or other personal consumer reports
3 -> Debt collection
4 -> Money transfer, virtual currency, or money service
5 -> Mortgage
6 -> Other financial service
7 -> Payday loan, title loan, or personal loan
8 -> Student loan
9 -> Vehicle loan or lease


In [14]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 16742
Testing samples: 4186


In [15]:
# 5. Experiment Function

def run_logistic_regression_experiment(
    data,
    text_column,
    max_features,
    test_size=0.20,
    random_state=42
):
    """
    Trains Logistic Regression on one text column and one TF-IDF setting.
    Returns metrics, model pipeline, label encoder, and test predictions.
    """

    experiment_df = data[[text_column, "category"]].dropna().copy()

    experiment_df[text_column] = experiment_df[text_column].astype(str)
    experiment_df["category"] = experiment_df["category"].astype(str)

    experiment_df = experiment_df[experiment_df[text_column].str.strip() != ""]
    experiment_df = experiment_df.reset_index(drop=True)

    X = experiment_df[text_column]
    y = experiment_df["category"]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=test_size,
        random_state=random_state,
        stratify=y_encoded
    )

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.90,
            sublinear_tf=True,
            strip_accents="unicode"
        )),
        ("model", LogisticRegression(
            C=2.0,
            max_iter=3000,
            solver="lbfgs",
            class_weight="balanced",
            n_jobs=-1,
            random_state=random_state
        ))
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    result = {
        "text_version": text_column,
        "max_features": max_features,
        "min_samples_per_class": MIN_SAMPLES_PER_CLASS,
        "num_rows": experiment_df.shape[0],
        "num_classes": len(label_encoder.classes_),
        "accuracy": round(accuracy, 6),
        "macro_f1": round(macro_f1, 6),
        "weighted_f1": round(weighted_f1, 6)
    }

    test_results = pd.DataFrame({
        "text": X_test.values,
        "actual_encoded": y_test,
        "predicted_encoded": y_pred
    })

    test_results["actual_category"] = label_encoder.inverse_transform(test_results["actual_encoded"])
    test_results["predicted_category"] = label_encoder.inverse_transform(test_results["predicted_encoded"])

    return result, pipeline, label_encoder, test_results

In [18]:

# Create Final Model DataFrame

# Check required columns
required_columns = ["complaint_text", "clean_text", "category"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Keep only required columns for model training
df_model = df[["clean_text", "category"]].copy()

# Remove missing values
df_model = df_model.dropna(subset=["clean_text", "category"])

# Convert columns to string
df_model["clean_text"] = df_model["clean_text"].astype(str)
df_model["category"] = df_model["category"].astype(str)

# Remove empty cleaned complaints
df_model = df_model[df_model["clean_text"].str.strip() != ""]

# Remove rare categories
MIN_SAMPLES_PER_CLASS = 50

category_counts = df_model["category"].value_counts()

valid_categories = category_counts[
    category_counts >= MIN_SAMPLES_PER_CLASS
].index

df_model = df_model[df_model["category"].isin(valid_categories)]

# Reset index
df_model = df_model.reset_index(drop=True)

print("df_model created successfully")
print("Final model data shape:", df_model.shape)
print("Number of categories:", df_model["category"].nunique())
print("\nCategory distribution:")
print(df_model["category"].value_counts())

df_model created successfully
Final model data shape: (20874, 2)
Number of categories: 8

Category distribution:
category
Credit card or prepaid card                                                     7099
Checking or savings account                                                     5936
Mortgage                                                                        3246
Credit reporting, credit repair services, or other personal consumer reports    2032
Debt collection                                                                  930
Money transfer, virtual currency, or money service                               853
Vehicle loan or lease                                                            638
Student loan                                                                     140
Name: count, dtype: int64


In [19]:
# Run All Experiments

text_versions = [
    "clean_text"
]

tfidf_feature_options = [
    10000,
    20000,
    30000
]

all_results = []
saved_objects = {}

for text_col in text_versions:
    for max_feat in tfidf_feature_options:
        print(f"Training: {text_col} | max_features={max_feat}")

        result, pipeline, label_encoder, test_results = run_logistic_regression_experiment(
            data=df_model,
            text_column=text_col,
            max_features=max_feat
        )

        all_results.append(result)

        key = f"{text_col}_tfidf_{max_feat}"

        saved_objects[key] = {
            "pipeline": pipeline,
            "label_encoder": label_encoder,
            "test_results": test_results
        }

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by=["weighted_f1", "macro_f1", "accuracy"],
    ascending=False
).reset_index(drop=True)

results_df

Training: clean_text | max_features=10000
Training: clean_text | max_features=20000
Training: clean_text | max_features=30000


,text_version,max_features,min_samples_per_class,num_rows,num_classes,accuracy,macro_f1,weighted_f1
0,clean_text,30000,50,20874,8,0.818683,0.737771,0.822284
1,clean_text,20000,50,20874,8,0.817006,0.737756,0.821066
2,clean_text,10000,50,20874,8,0.811976,0.727386,0.816770


In [20]:
#  Save Experiment Results

results_file = os.path.join(REPORT_DIR, "03_data_feature_experiment_results.csv")

results_df.to_csv(results_file, index=False)

print("Experiment results saved at:")
print(results_file)

results_df

Experiment results saved at:
C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\reports\03_data_feature_experiment_results.csv


,text_version,max_features,min_samples_per_class,num_rows,num_classes,accuracy,macro_f1,weighted_f1
0,clean_text,30000,50,20874,8,0.818683,0.737771,0.822284
1,clean_text,20000,50,20874,8,0.817006,0.737756,0.821066
2,clean_text,10000,50,20874,8,0.811976,0.727386,0.816770


In [21]:
# Select Best Experiment

best_row = results_df.iloc[0]

best_text_version = best_row["text_version"]
best_max_features = best_row["max_features"]

best_key = f"{best_text_version}_tfidf_{best_max_features}"

best_pipeline = saved_objects[best_key]["pipeline"]
best_label_encoder = saved_objects[best_key]["label_encoder"]
best_test_results = saved_objects[best_key]["test_results"]

print("Best experiment:")
print(best_row)

print("\nBest key:", best_key)

Best experiment:
text_version             clean_text
max_features                  30000
min_samples_per_class            50
num_rows                      20874
num_classes                       8
accuracy                   0.818683
macro_f1                   0.737771
weighted_f1                0.822284
Name: 0, dtype: object

Best key: clean_text_tfidf_30000


In [22]:
# Final Classification Report

y_test = best_test_results["actual_encoded"]
y_pred = best_test_results["predicted_encoded"]

report = classification_report(
    y_test,
    y_pred,
    target_names=best_label_encoder.classes_
)

print(report)

classification_report_file = os.path.join(REPORT_DIR, "03_best_classification_report.txt")

with open(classification_report_file, "w", encoding="utf-8") as f:
    f.write("Best Experiment\n")
    f.write(str(best_row))
    f.write("\n\nClassification Report\n")
    f.write(report)

print("Classification report saved at:")
print(classification_report_file)


# Save Classification Report as CSV for Streamlit UI

report_dict = classification_report(
    y_test,
    y_pred,
    target_names=best_label_encoder.classes_,
    output_dict=True
)

report_df = pd.DataFrame(report_dict).transpose()

classification_report_csv_file = os.path.join(
    REPORT_DIR,
    "03_best_classification_report.csv"
)

report_df.to_csv(classification_report_csv_file, index=True)

print("CSV classification report saved at:")
print(classification_report_csv_file)

                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.88      0.86      0.87      1187
                                                 Credit card or prepaid card       0.90      0.83      0.86      1420
Credit reporting, credit repair services, or other personal consumer reports       0.61      0.66      0.63       406
                                                             Debt collection       0.51      0.60      0.55       186
                          Money transfer, virtual currency, or money service       0.58      0.71      0.64       171
                                                                    Mortgage       0.91      0.92      0.91       649
                                                                Student loan       0.73      0.68      0.70        28
                                                       

In [23]:
# Save Misclassified Samples

misclassified = best_test_results[
    best_test_results["actual_category"] != best_test_results["predicted_category"]
].copy()

misclassified_file = os.path.join(REPORT_DIR, "03_misclassified_samples.csv")

misclassified.to_csv(misclassified_file, index=False, encoding="utf-8")

print("Misclassified samples saved at:")
print(misclassified_file)

print("Total test samples:", best_test_results.shape[0])
print("Total misclassified:", misclassified.shape[0])

misclassified.head(20)

Misclassified samples saved at:
C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\reports\03_misclassified_samples.csv
Total test samples: 4175
Total misclassified: 757


,text,actual_encoded,predicted_encoded,actual_category,predicted_category
7,chase xxxx provide full payment history even request filing complaint loan servicer xxxx,6,5,Student loan,Mortgage
25,follow timeline transpire xxxx xxxx present xxxx xxxx told xxxx lose job almost year xxxx xxxx xxxx xxxx last day apply job everywhere different career job site xxxx xxxx contact job company call ...,1,4,Credit card or prepaid card,"Money transfer, virtual currency, or money service"
43,go xxxx xxxx xxxx xxxx year xxxx xxxx get serve xxxx different bill payment later consolidated loan advise representative work company make payment taught principle young age make minimum payment ...,6,3,Student loan,Debt collection
55,chase manhatten mortgage corporation xxxx xxxx xxxx xxxx xxxx california xxxx chase private bank client bank xxxx telephone xxxx loan xxxx xxxx xxxx dollar state issue xxxx certification satisfact...,5,3,Mortgage,Debt collection
75,name xxxx xxxx desperate need help xxxx refinance home title company complete transaction never record discharge mortgage paid refinance also refinance time since back make mention process sell ho...,0,5,Checking or savings account,Mortgage
84,cruise xxxx xxxx try use chase debit card onxx xxxx pay bill cruise block fraud call take different call cellular company charge make call call chase explain fair block transaction manager xxxx sp...,0,4,Checking or savings account,"Money transfer, virtual currency, or money service"
87,account open xxxx credit limit plus unemployment xxxx insurance paid time till xxxx become ill work account balance contact insurance result xxxx contact chase bank regard closing account agree al...,3,0,Debt collection,Checking or savings account
88,chase bank hard inquiry xxxx credit report authorize know organization inquiry credit report fraud alert attachment credit report contact anyone try apply credit never contact dispute since xxxx x...,1,2,Credit card or prepaid card,"Credit reporting, credit repair services, or other personal consumer reports"
102,recently try write check xxxx another company check deny advise information report xxxx call xxxx advise morgan chase reporting information account chase never account call advise mom account expl...,0,3,Checking or savings account,Debt collection
113,short sale first mortgage mortgage first chase chase equity chase take xxxx chase equity chase kept charge late charge kept reporting debt kept sent short sale document many time chase never reply,2,5,"Credit reporting, credit repair services, or other personal consumer reports",Mortgage


In [24]:
# Save Best Model and Label Encoder

best_pipeline_file = os.path.join(MODEL_DIR, "best_complaint_classifier_pipeline.pkl")
best_encoder_file = os.path.join(MODEL_DIR, "best_label_encoder.pkl")

joblib.dump(best_pipeline, best_pipeline_file)
joblib.dump(best_label_encoder, best_encoder_file)

print("Best pipeline saved at:")
print(best_pipeline_file)

print("Best label encoder saved at:")
print(best_encoder_file)

Best pipeline saved at:
C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\models\best_complaint_classifier_pipeline.pkl
Best label encoder saved at:
C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\models\best_label_encoder.pkl


In [25]:
# Save Final Metrics JSON

final_metrics = {
    "best_text_version": str(best_text_version),
    "best_max_features": int(best_max_features),
    "min_samples_per_class": int(MIN_SAMPLES_PER_CLASS),
    "accuracy": float(best_row["accuracy"]),
    "macro_f1": float(best_row["macro_f1"]),
    "weighted_f1": float(best_row["weighted_f1"]),
    "num_rows": int(best_row["num_rows"]),
    "num_classes": int(best_row["num_classes"]),
    "algorithm": "LogisticRegression"
}

metrics_file = os.path.join(REPORT_DIR, "03_best_model_metrics.json")

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=4)

print("Final metrics saved at:")
print(metrics_file)

final_metrics

Final metrics saved at:
C:\Users\v-amaniyar\Downloads\Adhyayan_presentations\NLP_PROJECT\reports\03_best_model_metrics.json


{'best_text_version': 'clean_text',
 'best_max_features': 30000,
 'min_samples_per_class': 50,
 'accuracy': 0.818683,
 'macro_f1': 0.737771,
 'weighted_f1': 0.822284,
 'num_rows': 20874,
 'num_classes': 8,
 'algorithm': 'LogisticRegression'}

In [26]:
# 13. Test Prediction

def predict_category(model_text):
    prediction_encoded = best_pipeline.predict([model_text])[0]
    prediction_label = best_label_encoder.inverse_transform([prediction_encoded])[0]

    confidence = None

    if hasattr(best_pipeline.named_steps["model"], "predict_proba"):
        confidence = best_pipeline.predict_proba([model_text]).max()

    return prediction_label, confidence


sample_text = "unknown charge credit card bank not helping remove transaction"

predicted_category, confidence = predict_category(sample_text)

print("Predicted Category:", predicted_category)
print("Confidence:", round(confidence, 4))

Predicted Category: Credit card or prepaid card
Confidence: 0.599


In [27]:
# Load Saved Model and Test

loaded_pipeline = joblib.load(best_pipeline_file)
loaded_encoder = joblib.load(best_encoder_file)

sample_prediction_encoded = loaded_pipeline.predict([sample_text])[0]
sample_prediction_label = loaded_encoder.inverse_transform([sample_prediction_encoded])[0]

print("Loaded model prediction:", sample_prediction_label)

Loaded model prediction: Credit card or prepaid card
